In [22]:
"""
gptq_core.py

A minimal, from-scratch implementation of the GPTQ per-layer quantization
algorithm (Frantar et al., 2023), written to be simple to read and adapt --
not a copy of any production library. It only touches the layer you point it
at, so we use it on GPT-2's `c_attn` (the combined [W_Q | W_K | W_V] block).

Core idea (per layer):
  1. Accumulate a Hessian-like statistic H = 2 * X^T X from calibration
     activations X that flow into the layer.
  2. Quantize the weight one input-column at a time. After quantizing a
     column, immediately correct all *remaining* (not-yet-quantized)
     columns using the inverse Hessian, so later columns compensate for
     the error just introduced.
  3. The result is "fake-quantized": weights are rounded onto a low-bit
     grid, but stored back as ordinary float tensors -- no packing, no
     bit-unpacking needed later.
"""

import torch


def collect_hessian_via_hook(model: torch.nn.Module, module: torch.nn.Module,
                              calibration_batches, device) -> torch.Tensor:
    """
    Registers a forward pre-hook on `module` (e.g. one block's attn.c_attn)
    to capture its input activations, runs `calibration_batches` through the
    *whole model* in no_grad mode, and returns the accumulated Hessian
    H = 2 X^T X for that layer.

    `calibration_batches` should be an iterable of input_ids tensors of shape
    (batch, seq_len), already on `device`.

    Returns: H, a (d_in, d_in) double-precision tensor.
    """
    d_in = module.weight.shape[0]  # Conv1D weight is (in_features, out_features)
    H = torch.zeros(d_in, d_in, dtype=torch.float64, device=device)
    n_samples = [0]

    def _hook(mod, inputs):
        x = inputs[0].detach()
        x = x.reshape(-1, x.shape[-1]).to(torch.float64)  # (tokens, d_in)
        H.add_(2.0 * x.T @ x)
        n_samples[0] += x.shape[0]

    handle = module.register_forward_pre_hook(_hook)
    try:
        model.eval()
        with torch.no_grad():
            for input_ids in calibration_batches:
                model(input_ids.to(device))
    finally:
        handle.remove()

    if n_samples[0] > 0:
        H /= n_samples[0]
    return H


def _quantize_to_grid(w_col: torch.Tensor, scale: torch.Tensor, bits: int) -> torch.Tensor:
    """
    Symmetric per-output-row fake quantization of a single input-column
    (shape: d_out) using a fixed per-row scale (shape: d_out) computed
    up front from the original weight statistics.
    """
    qmax = 2 ** (bits - 1) - 1
    q = torch.clamp(torch.round(w_col / scale), -qmax, qmax)
    return q * scale


@torch.no_grad()
def gptq_quantize_layer(weight_in_out: torch.Tensor, H: torch.Tensor, bits: int = 4,
                         damp_percent: float = 0.01) -> torch.Tensor:
    """
    Quantizes a weight matrix in the (d_in, d_out) "Conv1D" convention
    (GPT-2 style: forward is x @ weight) using the GPTQ algorithm.

    weight_in_out: (d_in, d_out) float tensor -- e.g. c_attn.weight.data
    H: (d_in, d_in) Hessian from collect_hessian_via_hook
    bits: target bit-width
    damp_percent: Hessian damping factor for numerical stability (GPTQ default ~0.01)

    Returns the fake-quantized weight, same shape, and also writes it into
    weight_in_out in place.
    """
    device = weight_in_out.device
    # Work in the (d_out, d_in) "row = output channel" convention internally,
    # since GPTQ quantizes input-columns one at a time.
    W = weight_in_out.detach().clone().to(torch.float64).T.contiguous()  # (d_out, d_in)
    d_out, d_in = W.shape

    # Fixed per-output-row scale, computed once from the original weights.
    qmax = 2 ** (bits - 1) - 1
    scale = (W.abs().amax(dim=1, keepdim=True) / qmax).clamp(min=1e-8)  # (d_out, 1)

    # Damp and invert the Hessian.
    H = H.clone()
    mean_diag = H.diagonal().mean()
    H += damp_percent * mean_diag * torch.eye(d_in, dtype=torch.float64, device=device)
    H_inv = torch.linalg.inv(H)  # (d_in, d_in)

    for i in range(d_in):
        w_col = W[:, i]
        q_col = _quantize_to_grid(w_col, scale.squeeze(1), bits)
        err = (w_col - q_col) / H_inv[i, i]
        if i + 1 < d_in:
            W[:, i + 1:] -= torch.outer(err, H_inv[i, i + 1:])
        W[:, i] = q_col

    W_final = W.T.contiguous().to(weight_in_out.dtype)  # back to (d_in, d_out)
    weight_in_out.copy_(W_final)
    return W_final


In [23]:
import math
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset

#from gptq_core import collect_hessian_via_hook, gptq_quantize_layer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "gpt2"          # GPT-2 small, 124M params
SEQ_LEN = 512                # calibration / eval chunk length
N_CALIB_SAMPLES = 128        # standard GPTQ-paper convention
BITS = 4                     # target bit-width for this baseline run

In [24]:
def load_model_and_tokenizer():
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(DEVICE)
    model.eval()
    return model, tokenizer


def build_calibration_batches(tokenizer, n_samples=N_CALIB_SAMPLES, seq_len=SEQ_LEN):
    """
    Pulls `n_samples` chunks of `seq_len` tokens each from WikiText-2 train,
    as a list of (1, seq_len) input_id tensors -- the standard GPTQ-style
    calibration set.
    """

    raw = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="train")
    text = "\n\n".join(t for t in raw["text"] if t.strip())
    ids = tokenizer(text, return_tensors="pt").input_ids[0]

    batches = []
    stride = seq_len
    for i in range(n_samples):
        start = i * stride
        if start + seq_len > ids.shape[0]:
            break
        chunk = ids[start:start + seq_len].unsqueeze(0)
        batches.append(chunk)
    return batches

In [25]:
# ---------------------------------------------------------------------------
# Step 2 + 3: quantize every block's c_attn with GPTQ, then evaluate perplexity
# ---------------------------------------------------------------------------

def quantize_all_attention_blocks(model, calibration_batches):
    """
    For each transformer block, collects the Hessian for its c_attn layer
    and runs our GPTQ quantizer on it, in place. Blocks are handled
    independently from the *original* full-precision activations -- a
    known, documented simplification vs. the paper's fully sequential
    block-by-block quantization (which would re-run calibration through
    the partially-quantized model after each block). Simpler to implement,
    slightly less accurate; fine for a course-project baseline.
    """
    blocks = model.transformer.h
    for idx, block in enumerate(blocks):
        c_attn = block.attn.c_attn
        H = collect_hessian_via_hook(model, c_attn, calibration_batches, DEVICE)
        gptq_quantize_layer(c_attn.weight.data, H, bits=BITS)
        print(f"[week1] quantized block {idx}'s c_attn (Q|K|V) to {BITS} bits")


@torch.no_grad()

def evaluate_perplexity(model, tokenizer, max_length=1024, stride=512):
    """
+    Sliding-window perplexity on WikiText-2 test -- the standard method
+    (same one used in Hugging Face's own perplexity guide). Each window
+    uses up to `max_length` tokens of context, but loss is only counted on
+    the new (non-overlapping) `stride`-sized portion of each window, so
+    every scored token benefits from real preceding context instead of
+    starting cold like in non-overlapping chunking.
+
+    Smaller `stride` = more overlap = lower (better/more standard) PPL,
+    at the cost of more forward passes. stride=512 with max_length=1024
+    is a common, reasonable default; try stride=256 for an even tighter
+    (and slower) number.
+    """
    raw = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="test")
    text = "\n\n".join(t for t in raw["text"] if t.strip())
    #ids = tokenizer(text, return_tensors="pt").input_ids[0]
    ids = tokenizer(text, return_tensors="pt").input_ids.to(DEVICE)
    seq_len = ids.shape[1]

    model.eval()
    nll_sum = 0.0
    n_tokens = 0
    prev_end = 0
    for begin in range(0, seq_len, stride):
        end = min(begin + max_length, seq_len)
        trg_len = end - prev_end  # new tokens covered since the last window
        input_ids = ids[:, begin:end]
        target_ids = input_ids.clone()
        target_ids[:, :-trg_len] = -100  # only score the new portion

    #return math.exp(total_loss / total_tokens)
        out = model(input_ids, labels=target_ids)
        # out.loss is the mean over non-masked (target != -100) tokens
        num_valid = trg_len
        nll_sum += out.loss.item() * num_valid
        n_tokens += num_valid

        prev_end = end
        if end == seq_len:
            break

    return math.exp(nll_sum / n_tokens)



In [26]:
# ---------------------------------------------------------------------------
# Step 4: save the dequantized Q/K/V weights for Week 2
# ---------------------------------------------------------------------------

def split_qkv(c_attn_weight: torch.Tensor, n_embd: int):
    """c_attn.weight is (n_embd, 3*n_embd) = [W_Q | W_K | W_V] concatenated."""
    W_Q, W_K, W_V = c_attn_weight.split(n_embd, dim=1)
    return W_Q, W_K, W_V


def save_quantized_qkv(model, path="week1_gptq_qkv_init.pt"):
    n_embd = model.config.n_embd
    state = {}
    for idx, block in enumerate(model.transformer.h):
        W_Q, W_K, W_V = split_qkv(block.attn.c_attn.weight.data, n_embd)
        state[f"block_{idx}"] = {
            "W_Q": W_Q.clone().cpu(),
            "W_K": W_K.clone().cpu(),
            "W_V": W_V.clone().cpu(),
        }
    torch.save(state, path)
    print(f"[week1] saved GPTQ-initialized Q/K/V weights to {path}")


In [27]:
# ---------------------------------------------------------------------------
# Step 5: attention-output hook -- captures A(X), the input to c_proj
# ---------------------------------------------------------------------------

class AttentionOutputCapture:
    """
    Registers a forward pre-hook on every block's attn.c_proj. Its INPUT is
    exactly A(X) = softmax(QK^T/sqrt(d_k)) V, concatenated across heads,
    before the output projection W_O -- i.e. the attention output as
    defined in the project proposal.

    Usage:
        cap = AttentionOutputCapture(model)
        with torch.no_grad():
            model(input_ids)
        outputs = cap.outputs   # dict: block_idx -> tensor (batch, seq, n_embd)
        cap.remove()
    """

    def __init__(self, model):
        self.outputs = {}
        self.handles = []
        for idx, block in enumerate(model.transformer.h):
            handle = block.attn.c_proj.register_forward_pre_hook(self._make_hook(idx))
            self.handles.append(handle)

    def _make_hook(self, idx):
        def _hook(module, inputs):
            self.outputs[idx] = inputs[0]
        return _hook

    def remove(self):
        for h in self.handles:
            h.remove()



In [28]:
# ---------------------------------------------------------------------------
# Step 6: joint calibration loop -- SKELETON ONLY, completed in Week 2
# ---------------------------------------------------------------------------

def joint_calibration_skeleton(model_float, model_quant, calibration_batches):
    """
    Not implemented yet -- this is Week 2's task (with Student B, once
    JAB-Hessian is wired in). Left here so the interface is agreed on now:

      1. Wrap each block's c_attn weight in model_quant as an
         nn.Parameter with requires_grad=True, initialized from the
         GPTQ solution saved in Step 4.
      2. For each calibration batch:
           - forward model_float, capture target A(X) via AttentionOutputCapture
           - forward model_quant with a straight-through rounding op applied
             to c_attn.weight in the forward pass, capture Â(X)
           - loss = MSE(A(X), Â(X)), summed/averaged over blocks
           - loss.backward(); optimizer.step()
      3. Re-apply hard rounding at the end so the final weights are still
         on the quantization grid.

    TODO (Week 2): implement the straight-through estimator wrapper and
    the training loop above.
    """
    raise NotImplementedError("Week 2 task -- see docstring for the planned interface.")



In [ ]:
# ---------------------------------------------------------------------------
# main
# ---------------------------------------------------------------------------

if __name__ == "__main__":
    model, tokenizer = load_model_and_tokenizer()

    print("[week1] baseline perplexity (full precision):",
          evaluate_perplexity(model, tokenizer))

    calibration_batches = build_calibration_batches(tokenizer)
    quantize_all_attention_blocks(model, calibration_batches)

    print(f"[week1] R2 baseline perplexity (GPTQ, {BITS}-bit attention):",
          evaluate_perplexity(model, tokenizer))

    save_quantized_qkv(model)

    # quick sanity check that the attention-output hook works
    cap = AttentionOutputCapture(model)
    with torch.no_grad():
        model(calibration_batches[0].to(DEVICE))
    print("[week1] captured attention output shape for block 0:",
          cap.outputs[0].shape)
    cap.remove()


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (286177 > 1024). Running this sequence through the model will result in indexing errors


[week1] baseline perplexity (full precision): 24.35685268992344
[week1] quantized block 0's c_attn (Q|K|V) to 4 bits
[week1] quantized block 1's c_attn (Q|K|V) to 4 bits
[week1] quantized block 2's c_attn (Q|K|V) to 4 bits
[week1] quantized block 3's c_attn (Q|K|V) to 4 bits
